# Model Comparison

This file contains the model comparison performed on the ITSM datasets and was provided by Marc Hennig (mhennig@hm.edu).

# Environment

## Dependency installation

### PIP Dependencies

In [ ]:
!pip install ipdb scikit-learn==1.8.0 scikit-posthocs
!pip freeze > requirements.txt

## Dependency Imports

In [ ]:
# Python dependencies
import os
import re
import sys
import pathlib
from pathlib import Path
import locale
import shutil
import tempfile
import warnings
import math
from contextlib import contextmanager
import itertools

import random
import collections

from typing import List, Tuple, Dict, Union, Optional, Literal, Callable

import time
import datetime

import json

# Colab dependencies
from google.colab import files, drive, output

# Debugging
import ipdb
from tqdm.auto import tqdm

# Basic dependencies
import pandas as pd
import numpy as np
import scipy as sp
import statsmodels as sm
import statsmodels.stats
import statsmodels.stats.descriptivestats
import statsmodels.stats.multitest

# Plotting dependencies
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns

# Machine learning depenencies
import sklearn as sl
import sklearn.metrics

## Variables & Global Settings

In [ ]:
# Assign a random seed for reproduceability
RANDOM_STATE = 1337

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Show all Pandas columns
pd.set_option("display.max_columns", None)

# Set Matplotlib and Seaborn color scheme
plt.rcParams["image.cmap"] = "Blues"
sns.set_palette("Blues")

# Colab settings
output.enable_custom_widget_manager()

locale.getpreferredencoding = lambda: "UTF-8"

In [ ]:
# Google Drive folders
GDRIVE_INPUT_DIR = "/content/drive/My Drive/Colab Notebooks/Eventlogs"
GDRIVE_OUTPUT_DIR = "/content/drive/My Drive/Colab Notebooks/Results"

# Local Colab folders
UTIL_DIR = os.path.join(".", "Util")
DATA_DIR = os.path.join(".", "Data")
INPUT_DATA_DIR = os.path.join(DATA_DIR, "Input")
INPUT_DATA_BPIC2013_DIR = os.path.join(INPUT_DATA_DIR, "BPIC 2013")
INPUT_DATA_BPIC2014_DIR = os.path.join(INPUT_DATA_DIR, "BPIC 2014")
INTERIM_DATA_DIR = os.path.join(DATA_DIR, "Interim")
OUTPUT_DATA_DIR = os.path.join(DATA_DIR, "Output")

GRAPHIC_DIR = os.path.join(".", "Graphics")
MODEL_DIR = os.path.join(".", "Models")

Path(DATA_DIR).mkdir(exist_ok=True)
Path(INTERIM_DATA_DIR).mkdir(exist_ok=True)
Path(OUTPUT_DATA_DIR).mkdir(exist_ok=True)
Path(GRAPHIC_DIR).mkdir(exist_ok=True)
Path(MODEL_DIR).mkdir(exist_ok=True)

In [ ]:
GROUNDTRUTH_KEY = "y_true"
PREDICTIONS_KEY = "y_pred"

## Data Import

### A: Import from Google Drive

In [ ]:
#drive.mount("/content/drive")

#!cp -r "$GDRIVE_INPUT_DIR" "$INPUT_DATA_DIR"

#drive.flush_and_unmount()

### B: Upload from Local Machine

In [ ]:
uploaded = files.upload()
del uploaded

## Common Functions

In [ ]:
EVENTLOG_CASE = "case:concept:name"
EVENTLOG_ACTIVITY = "concept:name"
EVENTLOG_TIMESTAMP = "time:timestamp"
EVENTLOG_GROUP = "org:group"
EVENTLOG_RESOURCE = "org:resource"
EVENTLOG_ROLE = "org:role"
EVENTLOG_CASE_PREFIX = "case:"
EVENTLOG_LABEL_PREFIX = "label:"

EVENTLOG_LABEL_REM_TIME = f"{EVENTLOG_LABEL_PREFIX}time:timestamp:last"
EVENTLOG_LABEL_NEXT_ACT = f"{EVENTLOG_LABEL_PREFIX}concept:name:next"
EVENTLOG_LABEL_NEXT_TIME = f"{EVENTLOG_LABEL_PREFIX}time:timestamp:next"

EVENTLOG_FEAT_TIME_OF_YEAR_SUFFIX = ":timeofyear"
EVENTLOG_FEAT_TIME_OF_MONTH_SUFFIX = ":timeofmonth"
EVENTLOG_FEAT_TIME_OF_WEEK_SUFFIX = ":timeofweek"
EVENTLOG_FEAT_TIME_OF_DAY_SUFFIX = ":timeofday"
EVENTLOG_FEAT_TIME_ELAPSED_CYCLE_SUFFIX = ":elapsedcycle"
EVENTLOG_FEAT_TIME_ELAPSED_PREV_SUFFIX = ":elapsedprev"

TOKEN_PADDING = "[PAD]"
TOKEN_PADDING_NUM = 0
TOKEN_NA = "[NA]"
TOKEN_EOC = "[EOC]"

### Statistic & Visualization Functions

In [ ]:
def df_naive_regression_metrics(df_train: pd.DataFrame, df_test: pd.DataFrame, label_col: str, method: Literal['median', 'mean', 'mode'] = 'median'):
  if 'median' == method:
    y_pred = df_train[label_col].median()
  elif 'mean' == method:
    y_pred = df_train[label_col].mean()
  elif 'mode' == method:
    y_pred = df_train[label_col].mode()
  else:
    raise ValueError(f"Unknown method {method}")

  y_pred = np.full(df_test[label_col].size, y_pred)
  y_true = df_test[label_col].to_numpy()

  return evaluate_regression(y_true, y_pred)

def df_naive_classification_metrics(df_train: pd.DataFrame, df_test: pd.DataFrame, label_col: str):
  y_pred = df_train[EVENTLOG_LABEL_NEXT_ACT].mode().iloc[0]

  label_enc = sl.preprocessing.LabelEncoder()
  label_enc.fit(np.concatenate((df_train[label_col], df_test[label_col]), axis=None))

  y_pred = label_enc.transform(np.full(df_test[label_col].size, y_pred))
  y_true = label_enc.transform(df_test[label_col])

  return evaluate_classification(y_true, y_pred)

### File Functions

In [ ]:
def read_result_npz_files(
  base_folder: str,
  dataset: str,
  task: str = 'remaining_time',
  divider: str = '_',
  sort: bool = True,
  groundtruth_key = GROUNDTRUTH_KEY,
  predictions_key = PREDICTIONS_KEY
) -> dict[str, tuple[np.ndarray, np.ndarray]]:
  base_path = Path(base_folder)
  result = {}
  for f in base_path.glob(f"*{divider}{dataset}{divider}{task}.npz"):
    npz = np.load(f)
    regex = re.compile(f"^(?P<model>[^{divider}]+){divider}?(?P<model_variant>[^{divider}\\d]+)?{divider}?(?P<model_run>\\d+)?{divider}{dataset}{divider}{task}$")
    groups = re.match(regex, f.stem).groupdict()

    model_name = groups['model'] if groups.get('model_variant') is None else f"{groups['model']}{divider}{groups['model_variant']}"
    model_run = int(groups['model_run']) if groups.get('model_run') is not None else None

    if model_run is not None:
      if model_name not in result:
        result[model_name] = ([], [])
      result[model_name][0].append(npz[groundtruth_key])
      result[model_name][1].append(npz[predictions_key])
    else:
      result[model_name] = (npz[groundtruth_key], npz[predictions_key])

  if sort:
    result = {k: v for k, v in sorted(result.items(), key=lambda x: x[0])}

  print(f"Loaded {len(result)} results from {base_path}")
  print(result.keys())

  return result

def get_valid_filename(name):
  """
  Return the given string converted to a string that can be used for a clean filename. Remove leading and trailing spaces; convert other spaces to underscores; and remove anything that is not an alphanumeric, dash, underscore, or dot.
  """
  s = str(name).strip().replace(" ", "_")
  s = re.sub(r"(?u)[^-\w.]", "", s)
  return s

### Model Comparison

In [ ]:
@contextmanager
def suppress_metrics_warnings():
  warnings.filterwarnings('ignore', message='Precision is ill-defined and being set to')
  warnings.filterwarnings('ignore', message='Recall is ill-defined and being set to')
  warnings.filterwarnings('ignore', message='Only one class is present in y_true')
  warnings.filterwarnings('ignore', message='y_pred contains classes not in y_true')
  warnings.filterwarnings('ignore', message='The y_pred values do not sum to one. Make sure to pass probabilities.')
  warnings.filterwarnings('ignore', message='The y_prob values do not sum to one. Make sure to pass probabilities.')
  yield

In [ ]:
def per_sample_brier_loss(y_true: np.ndarray, y_prob: np.ndarray, labels: Optional[np.ndarray] = None) -> np.ndarray:
  if labels is None:
    labels = np.arange(y_prob.shape[1])

  y_onehot = np.zeros((len(y_true), len(labels)))
  y_onehot[np.arange(len(y_true)), y_true] = 1

  # Compute per-sample Brier score
  loss = np.sum((y_prob - y_onehot) ** 2, axis=1)

  return loss

def per_sample_crossentropy_loss(y_true: np.ndarray, y_prob: np.ndarray, labels: Optional[np.ndarray] = None, eps: float = 1e-15) -> np.ndarray:
  y_prob_true = y_prob[np.arange(len(y_true)), y_true]

  # Compute per-sample log-loss
  loss = -np.log(np.clip(y_prob_true, eps, 1.0))

  return loss

In [ ]:
def evaluate_regressions(mapping: Dict[str, tuple[np.ndarray, np.ndarray]]) -> pd.DataFrame:
  results = []
  for y_true, y_pred in mapping.values():
    if isinstance(y_pred, list) or y_pred.ndim > 1:
      result_metrics = [evaluate_regression(y_t, y_p) for y_t, y_p in zip(y_true, y_pred, strict=True)]
      results.append({key: np.mean([d[key] for d in result_metrics]) for key in result_metrics[0]})
    else:
      results.append(evaluate_regression(y_true, y_pred))

  return pd.DataFrame(results, index=mapping.keys()).sort_index()

def evaluate_classifications(mapping: Dict[str, tuple[np.ndarray, np.ndarray]]) -> pd.DataFrame:
  results = []
  for y_true, y_pred in mapping.values():
    if isinstance(y_pred, list) or y_pred.ndim > 2:
      result_metrics = [evaluate_classification(y_t, y_p) for y_t, y_p in zip(y_true, y_pred, strict=True)]
      results.append({key: np.mean([d[key] for d in result_metrics]) for key in result_metrics[0]})
    else:
      results.append(evaluate_classification(y_true, y_pred))

  return pd.DataFrame(results, index=mapping.keys()).sort_index()

def evaluate_regression(y_true: np.ndarray, y_pred: np.ndarray, return_df: bool = False) -> Union[Dict[str, float], pd.DataFrame]:
  if y_true.ndim > 1:
      y_true = np.squeeze(y_true, axis=1)
  if y_pred.ndim > 1:
      y_pred = np.squeeze(y_pred, axis=1)

  def logcosh_error(y_true, y_pred):
    error = np.subtract(y_pred, y_true)
    return np.mean(np.log((np.exp(error) + np.exp(-error))/2))

  eval = {
    'mae': float(sl.metrics.mean_absolute_error(y_true, y_pred)),
    'mse': float(sl.metrics.mean_squared_error(y_true, y_pred)),
    'rmse': float(sl.metrics.root_mean_squared_error(y_true, y_pred)),
    'mape': float(sl.metrics.mean_absolute_percentage_error(y_true, y_pred)),
    'medae': float(sl.metrics.median_absolute_error(y_true, y_pred)),
    'logcosh': float(logcosh_error(y_true, y_pred)),
    'max_error': float(sl.metrics.max_error(y_true, y_pred)),
  }

  if y_true.min() >= 0 and y_pred.min() >= 0:
    eval.update({
      'msle': float(sl.metrics.mean_squared_log_error(y_true, y_pred)),
      'rmsle': float(sl.metrics.root_mean_squared_log_error(y_true, y_pred)),
    })

  if return_df:
    return pd.DataFrame.from_dict(eval, orient='index', columns=["Value"])
  else:
    return eval

def evaluate_classification(y_true: np.ndarray, y_pred: np.ndarray, return_df: bool = False, k: int = 3, zero_division: str = 'warn') -> Union[Dict[str, float], pd.DataFrame]:
  y_true = np.squeeze(y_true)
  y_pred = np.squeeze(y_pred)
  y_pred_prob = None

  if y_true.ndim > 1:
    y_true = np.argmax(y_true, axis=1)
  if y_pred.ndim > 1:
    y_pred_prob = y_pred
    y_pred = np.argmax(y_pred, axis=1)

  with suppress_metrics_warnings():
    eval = {
      'accuracy': sl.metrics.accuracy_score(y_true, y_pred),
      'accuracy_balanced': sl.metrics.balanced_accuracy_score(y_true, y_pred),
      'accuracy_balanced_adjusted': sl.metrics.balanced_accuracy_score(y_true, y_pred, adjusted=True),
      'f1_micro': sl.metrics.f1_score(y_true, y_pred, average='micro', zero_division=zero_division),
      'f1_macro': sl.metrics.f1_score(y_true, y_pred, average='macro', zero_division=zero_division),
      'f1_weighted': sl.metrics.f1_score(y_true, y_pred, average='weighted', zero_division=zero_division),
      'precision_micro': sl.metrics.precision_score(y_true, y_pred, average='micro', zero_division=zero_division),
      'precision_macro': sl.metrics.precision_score(y_true, y_pred, average='macro', zero_division=zero_division),
      'precision_weighted': sl.metrics.precision_score(y_true, y_pred, average='weighted', zero_division=zero_division),
      'recall_micro': sl.metrics.recall_score(y_true, y_pred, average='micro'),
      'recall_macro': sl.metrics.recall_score(y_true, y_pred, average='macro'),
      'recall_weighted': sl.metrics.recall_score(y_true, y_pred, average='weighted'),
      'kappa': sl.metrics.cohen_kappa_score(y_true, y_pred),
      'kappa_weighted': sl.metrics.cohen_kappa_score(y_true, y_pred, weights='quadratic'),
      'mcc': sl.metrics.matthews_corrcoef(y_true, y_pred),
      'jaccard_micro': sl.metrics.jaccard_score(y_true, y_pred, average='micro', zero_division=zero_division),
      'jaccard_macro': sl.metrics.jaccard_score(y_true, y_pred, average='macro', zero_division=zero_division),
      'jaccard_weighted': sl.metrics.jaccard_score(y_true, y_pred, average='weighted', zero_division=zero_division),
      'hamming_loss': sl.metrics.hamming_loss(y_true, y_pred),
    }

    if y_pred_prob is not None:
      if len(np.unique(y_true)) > 2: # For multi-class probabilities
        eval.update({
          f'top{k}_accuracy': sl.metrics.top_k_accuracy_score(y_true, y_pred_prob, k=k, labels=range(y_pred_prob.shape[1])),
          'crossentropy_loss': sklearn.metrics.log_loss(y_true, y_pred_prob, normalize=True, labels=range(y_pred_prob.shape[1])),
          'brier_loss': sl.metrics.brier_score_loss(y_true, y_pred_prob, labels=range(y_pred_prob.shape[1])),
          #'roc_auc_ovr_micro' = sl.metrics.roc_auc_score(y_true, y_pred_prob, multi_class='ovr', average='micro', labels=range(y_pred_prob.shape[1])),
          'roc_auc_ovr_macro': sl.metrics.roc_auc_score(y_true, y_pred_prob, multi_class='ovr', average='macro', labels=range(y_pred_prob.shape[1])),
          'roc_auc_ovr_weighted': sl.metrics.roc_auc_score(y_true, y_pred_prob, multi_class='ovr', average='weighted', labels=range(y_pred_prob.shape[1])),
          #'roc_auc_ovo_micro' = sl.metrics.roc_auc_score(y_true, y_pred_prob, multi_class='ovo', average='micro', labels=range(y_pred_prob.shape[1])),
          'roc_auc_ovo_macro': sl.metrics.roc_auc_score(y_true, y_pred_prob, multi_class='ovo', average='macro', labels=range(y_pred_prob.shape[1])),
          'roc_auc_ovo_weighted': sl.metrics.roc_auc_score(y_true, y_pred_prob, multi_class='ovo', average='weighted', labels=range(y_pred_prob.shape[1])),
        })
      else: # For binary probabilities
        eval.update({
          'roc_auc': sl.metrics.roc_auc_score(y_true, y_pred_prob[:, 1]),
        })

  if return_df:
    return pd.DataFrame.from_dict(eval, orient='index', columns=["Value"])
  else:
    return eval

In [ ]:
def regression_paired_differences(y_true: np.ndarray, y_pred_model_1: np.ndarray, y_pred_model_2: np.ndarray, error_type: Literal['absolute', 'squared'] = 'absolute') -> np.ndarray:
  if 'absolute' == error_type:
    if isinstance(y_pred_model_1, list):
      model_1_errors = np.mean([np.abs(y_true - y_pred) for y_pred in y_pred_model_1], axis=0)
    else:
      model_1_errors = np.abs(y_true - y_pred_model_1)

    if isinstance(y_pred_model_2, list):
      model_2_errors = np.mean([np.abs(y_true - y_pred) for y_pred in y_pred_model_2], axis=0)
    else:
      model_2_errors = np.abs(y_true - y_pred_model_2)
  elif 'squared' == error_type:
    if isinstance(y_pred_model_1, list):
      model_1_errors = np.mean([np.square(y_true - y_pred) for y_pred in y_pred_model_1], axis=0)
    else:
      model_1_errors = np.square(y_true - y_pred_model_1)

    if isinstance(y_pred_model_2, list):
      model_2_errors = np.mean([np.square(y_true - y_pred) for y_pred in y_pred_model_2], axis=0)
    else:
      model_2_errors = np.square(y_true - y_pred_model_2)


  paired_differences = model_1_errors - model_2_errors # Positive value means model 2 is better
  return paired_differences

def classification_paired_differences(y_true: np.ndarray, y_pred_model_1: np.ndarray, y_pred_model_2: np.ndarray, error_type: Literal['prob', 'brier', 'crossentropy'] = 'prob') -> np.ndarray:
  if y_true.ndim > 1:
    y_true = np.argmax(y_true, axis=1)

  y_true = y_true.astype(int)

  if 'prob' == error_type:
    # Calculate paired differences in confidence for the true class
    if isinstance(y_pred_model_1, list):
      model_1_loss = np.mean([y_pred[np.arange(len(y_true)), y_true] for y_pred in y_pred_model_1], axis=0)
    else:
      model_1_loss = y_pred_model_1[np.arange(len(y_true)), y_true]

    if isinstance(y_pred_model_2, list):
      model_2_loss = np.mean([y_pred[np.arange(len(y_true)), y_true] for y_pred in y_pred_model_2], axis=0)
    else:
      model_2_loss = y_pred_model_2[np.arange(len(y_true)), y_true]

  elif 'brier' == error_type:
    if isinstance(y_pred_model_1, list):
      model_1_loss = np.mean([per_sample_brier_loss(y_true, y_pred) for y_pred in y_pred_model_1], axis=0)
    else:
      model_1_loss = per_sample_brier_loss(y_true, y_pred_model_1)

    if isinstance(y_pred_model_2, list):
      model_2_loss = np.mean([per_sample_brier_loss(y_true, y_pred) for y_pred in y_pred_model_2], axis=0)
    else:
      model_2_loss = per_sample_brier_loss(y_true, y_pred_model_2)

  elif 'crossentropy' == error_type:
    if isinstance(y_pred_model_1, list):
      model_1_loss = np.mean([per_sample_crossentropy_loss(y_true, y_pred) for y_pred in y_pred_model_1], axis=0)
    else:
      model_1_loss = per_sample_crossentropy_loss(y_true, y_pred_model_1)

    if isinstance(y_pred_model_2, list):
      model_2_loss = np.mean([per_sample_crossentropy_loss(y_true, y_pred) for y_pred in y_pred_model_2], axis=0)
    else:
      model_2_loss = per_sample_crossentropy_loss(y_true, y_pred_model_2)

  paired_differences = model_2_loss - model_1_loss

  return paired_differences

def regression_bootstrap_twosided_confidence_intervals(y_true: np.ndarray, y_pred_model_1: np.ndarray, y_pred_model_2: np.ndarray, metric: Callable[[np.ndarray, np.ndarray], np.ndarray] = sl.metrics.mean_absolute_error, n_iterations: int = 10000, alpha: float = 0.05) -> Tuple[float, float]:
  return bootstrap_twosided_confidence_intervals(y_true, y_pred_model_1, y_pred_model_2, metric, n_iterations, alpha)

def classification_bootstrap_twosided_confidence_intervals(y_true: np.ndarray, y_pred_model_1: np.ndarray, y_pred_model_2: np.ndarray, metric: Callable[[np.ndarray, np.ndarray], np.ndarray] = sl.metrics.accuracy_score, n_iterations: int = 10000, alpha: float = 0.05) -> Tuple[float, float]:
  return bootstrap_twosided_confidence_intervals(y_true, y_pred_model_1, y_pred_model_2, metric, n_iterations, alpha)

def bootstrap_twosided_confidence_intervals(y_true: np.ndarray, y_pred_model_1: np.ndarray, y_pred_model_2: np.ndarray, metric: Callable[[np.ndarray, np.ndarray], np.ndarray], n_iterations: int = 10000, alpha: float = 0.05) -> Tuple[float, float]:
  sample_size = len(y_true)

  boot_diffs = []
  for _ in range(n_iterations):
    # Resample indices with replacement
    indices = np.random.choice(range(sample_size), size=sample_size, replace=True)

    y_true_sample = y_true[indices]
    y_pred_1_sample = y_pred_model_1[indices]
    y_pred_2_sample = y_pred_model_2[indices]

    metric_1 = metric(y_true_sample, y_pred_1_sample)
    metric_2 = metric(y_true_sample, y_pred_2_sample)

    boot_diffs.append(metric_1 - metric_2)

  lower = np.percentile(boot_diffs, 100 * alpha / 2)
  upper = np.percentile(boot_diffs, 100 * (1 - alpha / 2))

  return float(lower), float(upper)

def bootstrap_paired_differences_twosided_confidence_interval(paired_differences: np.ndarray, n_iterations: int = 10000, alpha: float = 0.05) -> Tuple[float, float]:
  sample_size = len(paired_differences)

  boot_means = []
  for _ in range(n_iterations):
    # Resample indices with replacement
    indices = np.random.choice(paired_differences, size=sample_size, replace=True)

    paired_diff_sample = np.random.choice(paired_differences, size=len(paired_differences), replace=True)
    boot_means.append(np.mean(paired_diff_sample))

  lower = np.percentile(boot_means, 100 * alpha / 2)
  upper = np.percentile(boot_means, 100 * (1 - alpha / 2))

  return float(lower), float(upper)

def test_mcnemar(y_true: np.ndarray, y_pred_model_1: np.ndarray, y_pred_model_2: np.ndarray, exact: bool = True, return_df: bool = False) -> Union[Dict[str, float], pd.DataFrame]:
  if y_true.ndim > 1:
      y_true = np.argmax(y_true, axis=1)
  if y_pred_model_1.ndim > 1:
      y_pred_model_1 = np.argmax(y_pred_model_1, axis=1)
  if y_pred_model_2.ndim > 1:
      y_pred_model_2 = np.argmax(y_pred_model_2, axis=1)

  # Build the contingency table
  a = np.sum((y_true == y_pred_model_1) & (y_true == y_pred_model_2)) # both correct
  b = np.sum((y_true == y_pred_model_1) & (y_true != y_pred_model_2)) # m1 correct, m2 incorrect
  c = np.sum((y_true != y_pred_model_1) & (y_true == y_pred_model_2)) # m1 incorrect, m2 correct
  d = np.sum((y_true != y_pred_model_1) & (y_true != y_pred_model_2)) # both incorrect

  contingency_table = np.array([[a, b],
                                [c, d]], dtype=int)

  test_result = sm.stats.contingency_tables.mcnemar(contingency_table, exact=exact)

  # Effect sizes
  n = a + b + c + d
  odds_ratio = b / c if c != 0 else np.nan
  prop_diff = (b - c) / n

  eval = {
    "test_statistic": test_result.statistic,
    "p_value": test_result.pvalue,
    "contingency_table": contingency_table.tolist(),
    "proportion_difference": prop_diff,
    "odds_ratio": odds_ratio,
  }

  if return_df:
    return pd.DataFrame.from_dict(eval, orient='index', columns=["Value"])
  else:
    return eval

def test_wilcoxon_signed_rank(x: np.ndarray, y: Optional[np.ndarray] = None, return_df: bool = False) -> Union[Dict[str, float], pd.DataFrame]:
  if y is not None and len(x) != len(y):
    raise ValueError(f"x and y must have the same length, but have {len(x)}, {len(y)}")

  # Large sample size (n > 50) so use asymptotic approximation
  test_result = sp.stats.wilcoxon(x, y, alternative='two-sided', method='approx', correction=True)

  # Calculate Rosenthal's r effect size
  n = len(x)
  r = abs(test_result.zstatistic) / np.sqrt(n)

  eval = {
    "test_statistic": test_result.statistic,
    "p_value": test_result.pvalue,
    "z_score": test_result.zstatistic,
    "rosenthals_r": r,
  }

  if return_df:
    return pd.DataFrame.from_dict(eval, orient='index', columns=["Value"])
  else:
    return eval

def plot_p_value_matrix(df: pd.DataFrame, cbar: bool = True):
  mask = np.triu(np.ones_like(df.to_numpy()), k=1).astype(bool)
  #sns.set(font_scale=1.0)
  return sns.heatmap(df, annot=True, mask=mask, cmap='Blues_r', fmt='.3f', vmin=0, vmax=1, linewidths=0, square=True, cbar=cbar)

def test_sign(x: np.ndarray, y: Optional[np.ndarray] = None, return_df: bool = False) -> Union[Dict[str, float], pd.DataFrame]:
  if y is not None and len(x) != len(y):
    raise ValueError(f"x and y must have the same length, but have {len(x)}, {len(y)}")

  differences = x if y is None else x - y

  # Remove ties (zero differences)
  non_zero_diffs = differences[differences != 0]
  n_total = len(non_zero_diffs)

  if n_total == 0:
    raise ValueError("All differences are zero - cannot perform sign test")

  test_result = sm.stats.descriptivestats.sign_test(differences)

  # Calculate signed Cohen's g effect size
  n_positive = np.sum(non_zero_diffs > 0)
  proportion_positive = n_positive / n_total
  cohens_g = (2 * proportion_positive) - 1
  cohens_g_unsigned = g_unsigned = abs(proportion_positive - 0.5) / 0.5

  eval = {
    "test_statistic": test_result[0],
    "p_value": test_result[1],
    "cohens_g": cohens_g,
    "cohens_g_unsigned": cohens_g_unsigned,
  }

  if return_df:
    return pd.DataFrame.from_dict(eval, orient='index', columns=["Value"])
  else:
    return eval


def test_matrix_sign(y_true: Optional[np.ndarray] = None, prediction_type: Literal['regression', 'classification'] = 'regression', return_df: bool = False, value: str = 'p_value', **kwargs) -> Union[pd.DataFrame, np.ndarray]:
  arr = np.zeros((len(kwargs), len(kwargs)))
  for i, (k1, v1) in enumerate(kwargs.items()):
    if isinstance(v1, tuple):
      v1 = v1[1]
    for j, (k2, v2) in enumerate(kwargs.items()):
      if isinstance(v2, tuple):
        v2 = v2[1]
      if k1 == k2:
        arr[i][j] = 1
      else:
        if y_true is None:
          #arr[i][j] = sm.stats.descriptivestats.sign_test(v1 - v2)[1]
          arr[i][j] = test_sign(v1, v2, return_df=False)[value]
        else:
          paired_diffs = classification_paired_differences(y_true, v1, v2, error_type='crossentropy') if 'classification' == prediction_type else regression_paired_differences(y_true, v1, v2)
          arr[i][j] = test_sign(paired_diffs, return_df=False)[value]

  if return_df:
    return pd.DataFrame(arr, index=kwargs.keys(), columns=kwargs.keys())
  else:
    return arr

def test_matrix_wilcoxon_signed_rank(y_true: Optional[np.ndarray] = None, value: str = 'p_value', prediction_type: Literal['regression', 'classification'] = 'regression', return_df: bool = False, **kwargs) -> Union[pd.DataFrame, np.ndarray]:
  arr = np.zeros((len(kwargs), len(kwargs)))
  for i, (k1, v1) in enumerate(kwargs.items()):
    if isinstance(v1, tuple):
      v1 = v1[1]
    for j, (k2, v2) in enumerate(kwargs.items()):
      if isinstance(v2, tuple):
        v2 = v2[1]
      if y_true is None:
        #arr[i][j] = sp.stats.wilcoxon(v1, v2).pvalue
        arr[i][j] = test_wilcoxon_signed_rank(v1, v2, return_df=False)[value]
      else:
        paired_diffs = classification_paired_differences(y_true, v1, v2, error_type='brier') if 'classification' == prediction_type else regression_paired_differences(y_true, v1, v2)
        arr[i][j] = test_wilcoxon_signed_rank(paired_diffs, return_df=False)[value]

  if return_df:
    return pd.DataFrame(arr, index=kwargs.keys(), columns=kwargs.keys())
  else:
    return arr

def test_matrix_mann_whitney_u(return_df: bool = False, **kwargs) -> Union[pd.DataFrame, np.ndarray]:
  arr = np.zeros((len(kwargs), len(kwargs)))
  for i, (k1, v1) in enumerate(kwargs.items()):
    for j, (k2, v2) in enumerate(kwargs.items()):
      arr[i][j] = sp.stats.mannwhitneyu(v1, v2).pvalue

  if return_df:
    return pd.DataFrame(arr, index=kwargs.keys(), columns=kwargs.keys())
  else:
    return arr

def test_mann_whitney_u(a: np.ndarray, b: np.ndarray, return_df: bool = False) -> Union[Dict[str, float], pd.DataFrame]:
  test_result = sp.stats.mannwhitneyu(a, b, alternative='two-sided', use_continuity=True)

  eval = {
    "test_statistic": test_result.statistic,
    "p_value": test_result.pvalue,
  }

  if return_df:
    return pd.DataFrame.from_dict(eval, orient='index', columns=["Value"])
  else:
    return eval

def test_kruskal_wallis(*args, return_df: bool = False) -> Union[Dict[str, float], pd.DataFrame]:
  test_result = sp.stats.kruskal(*args)

  eval = {
    "test_statistic": test_result.statistic,
    "p_value": test_result.pvalue,
  }

  if return_df:
    return pd.DataFrame.from_dict(eval, orient='index', columns=["Value"])
  else:
    return eval

def test_friedman(*args, return_df: bool = False) -> Union[Dict[str, float], pd.DataFrame]:
  test_result = sp.stats.friedmanchisquare(*args)

  k = len(args)  # number of models
  n = len(args[0])  # number of samples

  kendalls_w = test_result.statistic / (n * (k - 1))

  eval = {
    "test_statistic": test_result.statistic,
    "p_value": test_result.pvalue,
    "kendalls_w": kendalls_w,
  }

  if return_df:
    return pd.DataFrame.from_dict(eval, orient='index', columns=["Value"])
  else:
    return eval

def interpret_effect_size(effect_size: float) -> Literal['negligible', 'small', 'medium', 'large']:
  if effect_size < 0.1:
    effect_interpretation = "negligible"
  elif effect_size < 0.3:
    effect_interpretation = "small"
  elif effect_size < 0.5:
    effect_interpretation = "medium"
  else:
    effect_interpretation = "large"

  return effect_interpretation

def visualize_paired_differences(
    paired_differences: np.ndarray,
    confidence_level: float = 0.95,
    ci_lower: Optional[float] = None,
    ci_upper: Optional[float] = None,
    name_model_1: str = "Model 1",
    name_model_2: str = "Model 2"
  ):
  mean_diff = np.mean(paired_differences)
  median_diff = np.median(paired_differences)

  # Create visualization
  plt.figure(figsize=(12, 6))

  # Main histogram
  plt.hist(paired_differences, bins='auto', density=True, alpha=0.7, color='skyblue', edgecolor='black')

  # Add kernel density estimate
  kde = sp.stats.gaussian_kde(paired_differences)
  x_range = np.linspace(min(paired_differences), max(paired_differences), 200)
  plt.plot(x_range, kde(x_range), 'r-', lw=2, label='KDE')

  # Reference lines
  plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5, label="No difference")
  plt.axvline(x=mean_diff, color='green', linestyle='-', label="Mean")
  plt.axvline(x=median_diff, color='blue', linestyle='-', label="Median")

  # Confidence interval
  if ci_lower is not None and ci_upper is not None:
    plt.axvspan(ci_lower, ci_upper, alpha=0.2, color='green', label=f'{confidence_level*100:.0f}% CI')

  plt.xlabel(f"Performance Difference ({name_model_1} - {name_model_2})")
  plt.ylabel("Density")
  plt.title("Distribution of Paired Differences")
  plt.legend()
  plt.grid(True, alpha=0.3)


In [ ]:
def correct_symmetric_test_matrix(df: Union[pd.DataFrame, np.ndarray], alpha: float = 0.05, method: str = 'holm', return_df: bool = False):
    if isinstance(df, pd.DataFrame):
        p_matrix = df.to_numpy()
    elif isinstance(df, np.ndarray):
        p_matrix = df
    else:
        raise ValueError("df must be a pandas DataFrame or a numpy array")

    if p_matrix.ndim != 2 or p_matrix.shape[0] != p_matrix.shape[1]:
        raise ValueError("p_matrix must be a square 2D array")

    n = p_matrix.shape[0]
    i_lower, j_lower = np.tril_indices(n, k=-1)   # exclude diagonal

    p_vals = p_matrix[i_lower, j_lower]

    # If only the diagonal exists (n=1), nothing to correct
    if len(p_vals) == 0:
        return p_matrix.copy()

    _, p_corr, _, _ = sm.stats.multitest.multipletests(p_vals, alpha=alpha, method=method)

    # Build the corrected symmetric array
    corrected = np.full_like(p_matrix, np.nan, dtype=float)
    corrected[i_lower, j_lower] = p_corr
    corrected[j_lower, i_lower] = p_corr

    if return_df:
        if isinstance(df, pd.DataFrame):
          return pd.DataFrame(corrected, index=df.index, columns=df.columns)
        else:
          return pd.DataFrame(corrected)

    return corrected

# Dataset: Incident management process enriched event log

In [ ]:
EVENT_LOG = "servicenow"

## Remaining Time Prediction

In [ ]:
PREDICTION_TASK = "remaining_time"

### Read Incident management process enriched event log V1

In [ ]:
results = read_result_npz_files("./", EVENT_LOG, PREDICTION_TASK)
results_pred = {k: y_pred for k, (y_true, y_pred) in results.items()}
results_true = [y_true for (y_true, y_pred) in results.values()][0]

### Calculate Performance Metrics

In [ ]:
evaluate_regressions(results)

### Perform Pairwise Wilcoxon Signed-Rank Test

In [ ]:
df = test_matrix_wilcoxon_signed_rank(**results_pred, y_true=results_true, return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_wilcoxon_signed_rank.svg"))

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_wilcoxon_signed_rank.svg"))

### Perform Pairwise Sign Test

In [ ]:
df = test_matrix_sign(**results_pred, y_true=results_true, return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_sign.svg"))

### Perform Groupwise Friedmann Test

In [ ]:
preds = [[pred] if not isinstance(pred, list) else pred for pred in results_pred.values()]
preds = itertools.chain(*preds)

test_friedman(*preds, return_df=True)

## Next Activity Prediction

In [ ]:
PREDICTION_TASK = "next_activity"

### Read Incident management process enriched event log V1

In [ ]:
results = read_result_npz_files("./", EVENT_LOG, PREDICTION_TASK)
results_pred = {k: y_pred for k, (y_true, y_pred) in results.items()}
results_true = [y_true for (y_true, y_pred) in results.values()][0]

### Calculate Performance Metrics

In [ ]:
evaluate_classifications(results)

### Perform Pairwise Wilcoxon Signed-Rank Test

In [ ]:
df = test_matrix_wilcoxon_signed_rank(**results_pred, y_true=results_true, prediction_type='classification', return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_wilcoxon_signed_rank.svg"))

### Perform Pairwise Sign Test

In [ ]:
df = test_matrix_sign(**results_pred, y_true=results_true, prediction_type='classification', return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_sign.svg"))

### Perform Groupwise Friedmann Test

In [ ]:
preds = [[pred] if not isinstance(pred, list) else pred for pred in results_pred.values()]
preds = itertools.chain(*preds)
preds = [pred[np.arange(len(results_true)), results_true.astype(int)] for pred in preds]

test_friedman(*preds, return_df=True)

# Dataset: Dataset belonging to the help desk log of an Italian Company

In [ ]:
EVENT_LOG = "italy"

## Remaining Time Prediction

In [ ]:
PREDICTION_TASK = "remaining_time"

### Read Dataset belonging to the help desk log of an Italian Company

In [ ]:
results = read_result_npz_files("./", EVENT_LOG, PREDICTION_TASK)
results_pred = {k: y_pred for k, (y_true, y_pred) in results.items()}
results_true = [y_true for (y_true, y_pred) in results.values()][0]

### Calculate Performance Metrics

In [ ]:
evaluate_regressions(results)

### Perform Pairwise Wilcoxon Signed-Rank Test

In [ ]:
df = test_matrix_wilcoxon_signed_rank(**results_pred, y_true=results_true, return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_wilcoxon_signed_rank.svg"))

### Perform Pairwise Sign Test

In [ ]:
df = test_matrix_sign(**results_pred, y_true=results_true, return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_sign.svg"))

### Perform Groupwise Friedmann Test

In [ ]:
preds = [[pred] if not isinstance(pred, list) else pred for pred in results_pred.values()]
preds = itertools.chain(*preds)

test_friedman(*preds, return_df=True)

## Next Activity Prediction

In [ ]:
PREDICTION_TASK = "next_activity"

### Read Dataset belonging to the help desk log of an Italian Company

In [ ]:
results = read_result_npz_files("./", EVENT_LOG, PREDICTION_TASK)
results_pred = {k: y_pred for k, (y_true, y_pred) in results.items()}
results_true = [y_true for (y_true, y_pred) in results.values()][0]

### Calculate Performance Metrics

In [ ]:
evaluate_classifications(results)

### Perform Pairwise Wilcoxon Signed-Rank Test

In [ ]:
df = test_matrix_wilcoxon_signed_rank(**results_pred, y_true=results_true, prediction_type='classification', return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_wilcoxon_signed_rank.svg"))

### Perform Pairwise Sign Test

In [ ]:
df = test_matrix_sign(**results_pred, y_true=results_true, prediction_type='classification', return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_sign.svg"))

### Perform Groupwise Friedmann Test

In [ ]:
preds = [[pred] if not isinstance(pred, list) else pred for pred in results_pred.values()]
preds = itertools.chain(*preds)
preds = [pred[np.arange(len(results_true)), results_true.astype(int)] for pred in preds]

test_friedman(*preds, return_df=True)

# Dataset: Helpdesk

In [ ]:
EVENT_LOG = "helpdesk"

## Remaining Time Prediction

In [ ]:
PREDICTION_TASK = "remaining_time"

### Read Helpdesk

In [ ]:
results = read_result_npz_files("./", EVENT_LOG, PREDICTION_TASK)
results_pred = {k: y_pred for k, (y_true, y_pred) in results.items()}
results_true = [y_true for (y_true, y_pred) in results.values()][0]

### Calculate Performance Metrics

In [ ]:
evaluate_regressions(results)

### Perform Pairwise Wilcoxon Signed-Rank Test

In [ ]:
df = test_matrix_wilcoxon_signed_rank(**results_pred, y_true=results_true, return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_wilcoxon_signed_rank.svg"))

### Perform Pairwise Sign Test

In [ ]:
df = test_matrix_sign(**results_pred, y_true=results_true, return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_sign.svg"))

### Perform Groupwise Friedmann Test

In [ ]:
preds = [[pred] if not isinstance(pred, list) else pred for pred in results_pred.values()]
preds = itertools.chain(*preds)

test_friedman(*preds, return_df=True)

## Next Activity Prediction

In [ ]:
PREDICTION_TASK = "next_activity"

### Read Helpdesk

In [ ]:
results = read_result_npz_files("./", EVENT_LOG, PREDICTION_TASK)
results_pred = {k: y_pred for k, (y_true, y_pred) in results.items()}
results_true = [y_true for (y_true, y_pred) in results.values()][0]

### Calculate Performance Metrics

In [ ]:
evaluate_classifications(results)

### Perform Pairwise Wilcoxon Signed-Rank Test

In [ ]:
df = test_matrix_wilcoxon_signed_rank(**results_pred, y_true=results_true, prediction_type='classification', return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_wilcoxon_signed_rank.svg"))

### Perform Pairwise Sign Test

In [ ]:
df = test_matrix_sign(**results_pred, y_true=results_true, prediction_type='classification', return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_sign.svg"))

### Perform Groupwise Friedmann Test

In [ ]:
preds = [[pred] if not isinstance(pred, list) else pred for pred in results_pred.values()]
preds = itertools.chain(*preds)
preds = [pred[np.arange(len(results_true)), results_true.astype(int)] for pred in preds]

test_friedman(*preds, return_df=True)

# Dataset: BPIC 2013

In [ ]:
EVENT_LOG = "bpic13"

## Remaining Time Prediction

In [ ]:
PREDICTION_TASK = "remaining_time"

### Read BPIC 2013

In [ ]:
results = read_result_npz_files("./", EVENT_LOG, PREDICTION_TASK)
results_pred = {k: y_pred for k, (y_true, y_pred) in results.items()}
results_true = [y_true for (y_true, y_pred) in results.values()][0]

### Calculate Performance Metrics

In [ ]:
evaluate_regressions(results)

### Perform Pairwise Wilcoxon Signed-Rank Test

In [ ]:
df = test_matrix_wilcoxon_signed_rank(**results_pred, y_true=results_true, return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_wilcoxon_signed_rank.svg"))

### Perform Pairwise Sign Test

In [ ]:
df = test_matrix_sign(**results_pred, y_true=results_true, return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_sign.svg"))

### Perform Groupwise Friedmann Test

In [ ]:
preds = [[pred] if not isinstance(pred, list) else pred for pred in results_pred.values()]
preds = itertools.chain(*preds)

test_friedman(*preds, return_df=True)

## Next Activity Prediction

In [ ]:
PREDICTION_TASK = "next_activity"

### Read BPIC 2013

In [ ]:
results = read_result_npz_files("./", EVENT_LOG, PREDICTION_TASK)
results_pred = {k: y_pred for k, (y_true, y_pred) in results.items()}
results_true = [y_true for (y_true, y_pred) in results.values()][0]

### Calculate Performance Metrics

In [ ]:
evaluate_classifications(results)

### Perform Pairwise Wilcoxon Signed-Rank Test

In [ ]:
df = test_matrix_wilcoxon_signed_rank(**results_pred, y_true=results_true, prediction_type='classification', return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_wilcoxon_signed_rank.svg"))

### Perform Pairwise Sign Test

In [ ]:
df = test_matrix_sign(**results_pred, y_true=results_true, prediction_type='classification', return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_sign.svg"))

### Perform Groupwise Friedmann Test

In [ ]:
preds = [[pred] if not isinstance(pred, list) else pred for pred in results_pred.values()]
preds = itertools.chain(*preds)
preds = [pred[np.arange(len(results_true)), results_true.astype(int)] for pred in preds]

test_friedman(*preds, return_df=True)

# Dataset: BPIC 2014

In [ ]:
EVENT_LOG = "bpic14"

## Remaining Time Prediction

In [ ]:
PREDICTION_TASK = "remaining_time"

### Read BPIC 2014

In [ ]:
results = read_result_npz_files("./", EVENT_LOG, PREDICTION_TASK)
results_pred = {k: y_pred for k, (y_true, y_pred) in results.items()}
results_true = [y_true for (y_true, y_pred) in results.values()][0]

### Calculate Performance Metrics

In [ ]:
evaluate_regressions(results)

### Perform Pairwise Wilcoxon Signed-Rank Test

In [ ]:
df = test_matrix_wilcoxon_signed_rank(**results_pred, y_true=results_true, return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_wilcoxon_signed_rank.svg"))

### Perform Pairwise Sign Test

In [ ]:
df = test_matrix_sign(**results_pred, y_true=results_true, return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_sign.svg"))

### Perform Groupwise Friedmann Test

In [ ]:
preds = [[pred] if not isinstance(pred, list) else pred for pred in results_pred.values()]
preds = itertools.chain(*preds)

test_friedman(*preds, return_df=True)

## Next Activity Prediction

In [ ]:
PREDICTION_TASK = "next_activity"

### Read BPIC 2014

In [ ]:
results = read_result_npz_files("./", EVENT_LOG, PREDICTION_TASK)
results_pred = {k: y_pred for k, (y_true, y_pred) in results.items()}
results_true = [y_true for (y_true, y_pred) in results.values()][0]

### Calculate Performance Metrics

In [ ]:
evaluate_classifications(results)

### Perform Pairwise Wilcoxon Signed-Rank Test

In [ ]:
df = test_matrix_wilcoxon_signed_rank(**results_pred, y_true=results_true, prediction_type='classification', return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_wilcoxon_signed_rank.svg"))

### Perform Pairwise Sign Test

In [ ]:
df = test_matrix_sign(**results_pred, y_true=results_true, prediction_type='classification', return_df=True)
df

In [ ]:
fig = plot_p_value_matrix(df, cbar=True)
fig.get_figure().savefig(os.path.join(GRAPHIC_DIR, f"{EVENT_LOG}_{PREDICTION_TASK}_heatmap_sign.svg"))

### Perform Groupwise Friedmann Test

In [ ]:
preds = [[pred] if not isinstance(pred, list) else pred for pred in results_pred.values()]
preds = itertools.chain(*preds)
preds = [pred[np.arange(len(results_true)), results_true.astype(int)] for pred in preds]

test_friedman(*preds, return_df=True)